In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

In [2]:
data = pd.read_csv('data/data_AuCd2.csv')
data.head()

,Sample_name,Auseed_diameter_ave_nm,Auseed_diameter_stdv_nm,Auseed_diameter_max_nm,Auseed_diameter_min_nm,Au_mmol,Cd_mmol,oleylamine_mL,oleic_acid_mL,heating_rate_C/min,holding_temperature_C,holding_time_h,ratio_AuCd,H,V,C,Class
0,Au,13.4,1.3,16.500476,2.689001,1.0,0.000000,0.0,0.0,0.000000,25,0.0,0.0,R,6.1,14.9,0
1,sam254_1.5h,13.4,1.3,16.500476,2.689001,0.5,1.002511,33.0,3.2,1.698113,271,1.5,53.4,Y,3.5,5.4,0
2,sam254_3h,13.4,1.3,16.500476,2.689001,0.5,1.002511,33.0,3.2,1.698113,271,3.0,57.8,RP,3.9,0.5,0
3,sam255_1.5h,13.4,1.3,16.500476,2.689001,0.5,0.501256,33.0,3.2,4.142857,270,1.5,0.0,YR,5.8,12.2,0
4,sam255_3h,13.4,1.3,16.500476,2.689001,0.5,0.501256,33.0,3.2,4.142857,270,3.0,0.0,RP,3.5,3.5,0


In [3]:
y = pd.DataFrame(data['Class'], columns=['Class'])
X = data.drop(columns=['Sample_name', 'ratio_AuCd', 'H', 'V', 'C', 'ratio_AuCd', 'Class'])

data_RS1 = []
data_PS1 = []
data_F11 = []
data_AS1 = []
data_RS2 = []
data_PS2 = []
data_F12 = []
data_AS2 = []
for i in range(0,10):
    #print('Run', i)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i, stratify=y)
    param_grid = {'max_depth':[4, 6],'max_leaf_nodes': [3, 7, 11], 'min_samples_leaf': [3, 5]}
    clf = GridSearchCV(HistGradientBoostingClassifier(random_state=0, max_bins=51), param_grid=param_grid, scoring='accuracy', cv=10, n_jobs=12)
    clf.fit(X_train,y_train['Class'])
    clf_best = clf.best_estimator_
    #print(clf.best_params_)
    y_pred1 = clf_best.predict(X_train)
    y_pred2 = clf_best.predict(X_test)
    #train
    AS1 = accuracy_score(y_train, y_pred1)
    RS1 = recall_score(y_train, y_pred1, average=None)
    PS1 = precision_score(y_train, y_pred1, average=None)
    F11 = f1_score(y_train, y_pred1, average=None)
    #test
    AS2 = accuracy_score(y_test, y_pred2)
    RS2 = recall_score(y_test, y_pred2, average=None)
    PS2 = precision_score(y_test, y_pred2, average=None)
    F12 = f1_score(y_test, y_pred2, average=None)
    #print(F12)
    
    data_RS1.append(RS1)
    data_PS1.append(PS1)
    data_F11.append(F11)
    data_AS1.append(AS1)
    data_RS2.append(RS2)
    data_PS2.append(PS2)
    data_F12.append(F12)
    data_AS2.append(AS2)
    #print(classification_report(y_test, y_pred2))


columns = ['Class_0', 'Class_1']    
data_RS_train = pd.DataFrame(data=data_RS1, columns=columns)
data_PS_train = pd.DataFrame(data=data_PS1, columns=columns) 
data_F1_train = pd.DataFrame(data=data_F11, columns=columns) 
data_AS_train = pd.DataFrame(data=data_AS1, columns=['AS_train'])
data_RS_test = pd.DataFrame(data=data_RS2, columns=columns)
data_PS_test = pd.DataFrame(data=data_PS2, columns=columns) 
data_F1_test = pd.DataFrame(data=data_F12, columns=columns)  
data_AS_test = pd.DataFrame(data=data_AS2, columns=['AS_test'])        
data_AS = pd.concat([data_AS_train, data_AS_test], axis=1, join='inner')
    
data_RS_train.to_csv('result/ML/HGB/RS_train.csv')
data_PS_train.to_csv('result/ML/HGB/PS_train.csv')
data_F1_train.to_csv('result/ML/HGB/F1_train.csv')
data_RS_test.to_csv('result/ML/HGB/RS_test.csv')
data_PS_test.to_csv('result/ML/HGB/PS_test.csv')
data_F1_test.to_csv('result/ML/HGB/F1_test.csv')
data_AS.to_csv('result/ML/HGB/AS.csv')

print(f'Accuracy : Mean={data_AS_test["AS_test"].mean():.3f}, Std={data_AS_test["AS_test"].std():.3f}')
print(f"Recall : Mean={[f'{val:.3f}' for val in data_RS_test.mean(axis=0).values]}, Std={[f'{val:.3f}' for val in data_RS_test.std(axis=0).values]}")
print(f"Precision : Mean={[f'{val:.3f}' for val in data_PS_test.mean(axis=0).values]}, Std={[f'{val:.3f}' for val in data_PS_test.std(axis=0).values]}")
print(f"F1 Score : Mean={[f'{val:.3f}' for val in data_F1_test.mean(axis=0).values]}, Std={[f'{val:.3f}' for val in data_F1_test.std(axis=0).values]}")

Accuracy : Mean=0.893, Std=0.038
Recall : Mean=['0.913', '0.800'], Std=['0.050', '0.249']
Precision : Mean=['0.959', '0.687'], Std=['0.051', '0.139']
F1 Score : Mean=['0.933', '0.714'], Std=['0.023', '0.134']
